In [ ]:
import os
import re
import math
import random
from tqdm import tqdm
import torch
import torch.nn.functional as f
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from datasets import load_from_disk
from torch.utils.data import Dataset, DataLoader

In [ ]:
#提取真实答案
def extract_hash_answer(text: str) -> str | None:
    try:
        return text.split("####")[1].strip()
    except IndexError:
        return None

# 提取模型生成答案
def extract_xml_answer(text: str) -> str:
    try:
        return text.split("<answer>")[-1].split("</answer>")[0].strip()
    except IndexError:
        return ""

In [ ]:
def compute_format_score(batch_responses):
    pattern = r"^<reasoning>(?:(?!</reasoning>).)*</reasoning>\n<answer>(?:(?!</answer>).)*</answer>$"
    matchs = [bool(re.match(pattern, g_a)) for g_a in batch_responses]
    format_scores = [1.0 if match else 0.0 for match in matchs]
    return format_scores

def compute_reward(batch_answers, answers):
    reward_scores = [2.0 if g_a == a else 0.0 for g_a, a in zip(batch_answers, answers)]
    return reward_scores

In [ ]:
def get_lr(it, max_steps, warm_steps = None, max_lr=1e-5, min_lr=1e-6):
    warm_steps = int(0.1 * max_steps)

    # 预热时学习率线性增加
    if it < warm_steps:
        return max_lr * (it+1) / warm_steps
    
    # 确保最大步数之后，学习率不崩
    if it > max_steps:
        return min_lr

    # 余弦退火
    decay_ratio = (it - warm_steps) / (max_steps - warm_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)  

In [ ]:
SYSTEM_PROMPT = (
    """
    A conversation between User and Assistant. The user asks a question, and the Assistant solves it.
    The assistant first thinks about the reasoning process in the mind and then provides the user
    with the answer. The reasoning process and answer are enclosed within <reasoning> </reasoning> and
    <answer> </answer> tags, respectively.
    Example:
    <reasoning> ... </reasoning>
    <answer>42</answer>
    """
)

TASK_SPECIFIC_INSTRUCTIONS = "The answer must be a single integer."

In [ ]:
dist.init_process_group(backend="nccl")
world_size = dist.get_world_size()
rank = dist.get_rank()
# Determine local GPU id for this process
local_rank = int(os.environ.get("LOCAL_RANK", 0))
torch.cuda.set_device(local_rank)
master_process = rank == 0

# 为每个GPU生成设置不同的随机种子，防止GRPO过程中优势为零
seed = 42 + rank
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# --------------------------------------
# Hyperparameters
# --------------------------------------
max_grad_norm = 0.1          # For gradient clipping
ppo_clip_range = 0.2         # ε for PPO clipping
kl_coef = 0.005                # KL散度惩罚系数
batch_size = 4               # Prompts per batch per GPU
K = 4                        # GRPO每个prompt采样的数量
ppo_epochs = 4               # PPO update epochs per batch
max_new_tokens = 256         # Max tokens to generate per sample
num_epochs = 1               # Number of training epochs
weight_decay = 0.1  
initial_learning_rate = 1e-5

In [ ]:
model_name = "Your_favorite_model" # what I use: Llama-3.2-1B-Instruct
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Ensure pad token is defined (use eos as pad)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 策略模型
policy_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to(local_rank)

policy_model.train()

# 参考模型
ref_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to(local_rank)

ref_model.eval()

for p in ref_model.parameters():
    p.requires_grad = False

policy_model = DDP(policy_model, device_ids=[local_rank], output_device=local_rank)

param_dict = {pn: p for pn, p in policy_model.named_parameters() if p.requires_grad}

decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': weight_decay},
    {'param': nodecay_params, 'weight_decay': 0.0}
]
optimizer = torch.optim.AdamW(optim_groups, lr = initial_learning_rate)
optimizer.zero_grad(set_to_none = True)
